# T2 — Procesamiento de Entrevistas
**MCP · EGADE Business School · Summer 2026**

Lee archivos `.txt` con formato `Nombre_1.txt`, `Nombre_2.txt`, `Nombre_3.txt` y genera una base de datos consolidada.

---

### Formato de archivos
```
Jairo_1.txt  →  respuesta de Jairo a P1
Jairo_2.txt  →  respuesta de Jairo a P2
Jairo_3.txt  →  respuesta de Jairo a P3
```
El contenido de cada archivo es solo el texto transcrito, sin formato especial.

In [ ]:
import pandas as pd
import os
from pathlib import Path
from collections import defaultdict

# ── CONFIGURACIÓN ──────────────────────────────────────────
CARPETA_ENTREVISTAS = Path("entrevistas")
ARCHIVO_SALIDA      = "T2_base_entrevistas.xlsx"

# Agrega aquí el rol y departamento de cada persona
# Clave: nombre exacto tal como aparece en el archivo (antes del _)
INFO_PERSONAS = {
    "Jairo": {"ROL": "Researcher", "Entrepreneurship": ""},
    # Agrega más personas aquí:
    # "Maria": {"ROL": "Director", "DEPARTAMENTO": "Marketing"},
}
# ───────────────────────────────────────────────────────────

if not CARPETA_ENTREVISTAS.exists():
    CARPETA_ENTREVISTAS.mkdir()
    print(f"📁 Carpeta '{CARPETA_ENTREVISTAS}' creada. Agrega los archivos .txt ahí.")
else:
    archivos = sorted(CARPETA_ENTREVISTAS.glob("*.txt"))
    print(f"📄 Archivos encontrados: {len(archivos)}")
    for a in archivos:
        print(f"   · {a.name}")

In [ ]:
# ── PARSEAR ARCHIVOS ───────────────────────────────────────
# Agrupa por nombre y asigna cada archivo a P1, P2, P3 u OTROS
datos = defaultdict(lambda: {"P1": "", "P2": "", "P3": "", "OTROS": ""})

for archivo in archivos:
    stem = archivo.stem  # e.g. "Jairo_1"
    partes = stem.rsplit("_", 1)  # ["Jairo", "1"]
    
    if len(partes) != 2:
        print(f"⚠️  Nombre no reconocido: {archivo.name} (formato esperado: Nombre_1.txt)")
        continue
    
    nombre, num = partes[0], partes[1]
    contenido = archivo.read_text(encoding="utf-8").strip()
    
    if num == "1":
        datos[nombre]["P1"] = contenido
    elif num == "2":
        datos[nombre]["P2"] = contenido
    elif num == "3":
        datos[nombre]["P3"] = contenido
    else:
        datos[nombre]["OTROS"] += f" [{num}] {contenido}"
    
    print(f"✅ {archivo.name} → {nombre} · P{num}")

print(f"\nPersonas encontradas: {list(datos.keys())}")

In [ ]:
# ── CONSTRUIR DATAFRAME ────────────────────────────────────
registros = []

for nombre, respuestas in datos.items():
    info = INFO_PERSONAS.get(nombre, {"ROL": "", "DEPARTAMENTO": ""})
    registro = {
        "NOMBRE":       nombre,
        "DEPARTAMENTO": info.get("DEPARTAMENTO", ""),
        "ROL":          info.get("ROL", ""),
        "P1":           respuestas["P1"],
        "P2":           respuestas["P2"],
        "P3":           respuestas["P3"],
        "OTROS":        respuestas["OTROS"],
    }
    registros.append(registro)

df = pd.DataFrame(registros)
print(f"Total entrevistados: {len(df)}")
df

In [ ]:
# ── EXPORTAR A EXCEL ───────────────────────────────────────
df.to_excel(ARCHIVO_SALIDA, index=False)
print(f"✅ Exportado: {ARCHIVO_SALIDA}")